# The Probability Domain — fast path only

**Companion notebook 02 for W43 — The Probability Domain.**
The flagship [The Probability Domain](https://artifocial.com/blog/probability-domain-2026-oct-22) offers a **fast path** for the
practitioner who wants the eval error bar and the SGD claim and will come back for the objects:
§0 → §2.2–2.4 → §3.3 → §4.2 → §5.1 + §5.4 → §7.6 → §8.2–8.4 → §12–13 — fourteen stops. This notebook runs **exactly the numbered
sections of `worked_numbers.py` that those fourteen stops quote, in the order they appear in the text, and nothing else**:
`§1`, `§2`, `§6` (§0) · `§20` (§2.4) · `§33`, `§18` (§3.3) · `§5`, `§49` (§4.2) · `§7` (§5.1) · `§8` (§5.4) · `§15`, `§35`, `§42` (§7.6) ·
`§22`, `§37` (§8.2) · `§21`, `§34`, `§40` (§8.4). Five of the stops — §2.2, §2.3, §8.3, §12 and §13 — quote no script section, and
each has a one-line markdown cell saying so.

Every code cell is a script section **verbatim**, and `sec()` restores the generator state `worked_numbers.py` had when it reached
that section (recorded in `rng_states.json`), so every cell prints the article's numbers whether the notebook is run top to bottom
or opened at one cell. `Phi` is defined in the setup cell; `§35` defines its own `binom_two_sided`; no fast-path section needs
`fit_logreg`. Pure NumPy, CPU-only, float64. The full treatment of each section, with figures, is in
[`00_probability_from_scratch.ipynb`](./00_probability_from_scratch.ipynb) (`§1`–`§14`) and
[`01_concentration_and_chains.ipynb`](./01_concentration_and_chains.ipynb) (`§15`–`§49`).

In [1]:
import sys, platform, json, math
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from math import erf, sqrt, log, exp, pi, comb

matplotlib.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 110, "font.size": 10, "figure.figsize": (7.2, 3.6),
    "axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#52514e", "axes.labelcolor": "#0b0b0b", "xtick.color": "#52514e",
    "ytick.color": "#52514e", "figure.autolayout": True, "legend.frameon": False,
})
C1, C2, C3, C4, C5 = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"
np.set_printoptions(precision=4, suppress=True)

# Every section below restores the exact generator state the worked-numbers script had at that
# section, recorded once by capture_rng_states.py. The seeds are 20261022 (flagship, "§N") and
# 20261029 (companions, "A?" / "B?"). A cell therefore prints the article's numbers whether you run
# the notebook top to bottom or jump straight to it. Every helper and sample size a cell needs is defined above, so
# a numbered cell runs on its own immediately after this one (checked by check_cell_isolation.py).
STATES = json.load(open("rng_states.json"))
rng = np.random.default_rng(0)

def Phi(x):
    return 0.5 * (1 + erf(x / sqrt(2)))

# Shared names, so that ANY numbered cell runs on its own right after this cell:
N = 1_000_000          # the companion script sets N once, in A3, and A6/A11 reuse it; the flagship's §1/§2 set their own
from math import lgamma
def binom_pmf(n, p):   # defined in the companion script's B3, used again by B5 (copied verbatim)
    k = np.arange(n+1); lg = np.array([lgamma(n+1) - lgamma(i+1) - lgamma(n-i+1) for i in k])
    return np.exp(lg + k*np.log(p) + (n-k)*np.log1p(-p))
def fit_logreg(X, y, iters=25, l2=1e-3):   # defined in the flagship's §4 (notebook 00), used by §38 and §46 (copied verbatim)
    w = np.zeros(X.shape[1])
    for _ in range(iters):
        p = 1/(1+np.exp(-X@w)); z = (y+1)/2
        g = X.T@(p-z)/len(y) + l2*w
        H = (X.T*(p*(1-p)))@X/len(y) + l2*np.eye(X.shape[1])
        w -= np.linalg.solve(H, g)
    return w

# Standard-error helpers used by the SE lines (copied verbatim from worked_numbers.py)
def se_var(a):
    """SE of the sample variance of an iid sample: sqrt((m4 - s^4)/n), m4 the sample fourth central moment"""
    a = np.asarray(a, float); c = a - a.mean(); s2 = np.mean(c*c)
    return sqrt(max(np.mean(c**4) - s2*s2, 0.0)/len(a))
def se_sd(a):
    """SE of the sample SD: SE(var)/(2 sd)"""
    a = np.asarray(a, float); s = a.std()
    return se_var(a)/(2*s) if s > 0 else 0.0
def se_median(a):
    """distribution-free SE of a sample median: half the width of the order-statistic ~68% interval, ranks n/2 -/+ sqrt(n)/2"""
    a = np.sort(np.asarray(a, float)); n = len(a)
    if n < 4: return float('nan')
    lo, hi = max(int(math.floor(n/2 - sqrt(n)/2)), 0), min(int(math.ceil(n/2 + sqrt(n)/2)), n - 1)
    return (a[hi] - a[lo])/2
def jackknife_kurt(y):
    """delete-one jackknife SE of the sample excess kurtosis (sums updated in O(n); no resampling, no draws)"""
    y = np.asarray(y, float); n = len(y); m = n - 1
    S = [np.sum(y**k) for k in (1, 2, 3, 4)]
    r1 = (S[0] - y)/m; r2 = (S[1] - y**2)/m; r3 = (S[2] - y**3)/m; r4 = (S[3] - y**4)/m
    mu2 = r2 - r1**2; mu4 = r4 - 4*r3*r1 + 6*r2*r1**2 - 3*r1**4
    ki = mu4/mu2**2 - 3
    return sqrt((n - 1)/n*np.sum((ki - ki.mean())**2))

# ----------------------------------------------------------------------------------------------

def sec(tag, title):
    key = "flagship" if isinstance(tag, int) else "companions"
    rng.bit_generator.state = STATES[key][str(tag)]
    print(f"\n=== {'§' if key == 'flagship' else ''}{tag} {title} ===")

print("Python     ", sys.version.split()[0], "on", platform.system(), platform.machine())
print("NumPy      ", np.__version__)
print("Matplotlib ", matplotlib.__version__)
print("Sections   ", {k: len(v) for k, v in STATES.items()}, "generator states on file")

Python      3.14.5 on Darwin x86_64
NumPy       2.5.3
Matplotlib  3.11.1
Sections    {'flagship': 53, 'companions': 37} generator states on file


---
## Stop 1 — §0. Three puzzles that are one puzzle

Three puzzles whose "obvious" answer is wrong for the same reason — the measure was never named — and the flagship opens with
all three in numbers: Bertrand's chord (`§1`), Monty Hall (`§2`) and Borel–Kolmogorov (`§6`).

### §1 — Bertrand's chord: three "uniform" measures, three answers

A random chord of the unit circle is longer than the inscribed triangle's side with probability $1/3$, $1/2$ or $1/4$ depending on
which of three honest procedures "uniform" names. The question was never ambiguous; the measure was unspecified.

In [2]:
sec(1, "Bertrand's chord: three 'uniform' measures, three answers (unit circle, side = sqrt(3))")
N = 1_000_000
# (a) random endpoints
t1, t2 = rng.uniform(0, 2 * pi, N), rng.uniform(0, 2 * pi, N)
L = 2 * np.abs(np.sin((t1 - t2) / 2))
pa = np.mean(L > sqrt(3))
print(f"(a) two uniform endpoints        P(L > sqrt3) = {pa:.4f}   (theory 1/3 = {1/3:.4f})")
# (b) random radius point: distance from centre uniform on [0,1]
d = rng.uniform(0, 1, N)
L = 2 * np.sqrt(1 - d**2)
pb = np.mean(L > sqrt(3))
print(f"(b) uniform distance to centre   P(L > sqrt3) = {pb:.4f}   (theory 1/2 = {0.5:.4f})")
# (c) random midpoint uniform in the disc
r = np.sqrt(rng.uniform(0, 1, N))
L = 2 * np.sqrt(1 - r**2)
pc = np.mean(L > sqrt(3))
print(f"(c) uniform midpoint in the disc P(L > sqrt3) = {pc:.4f}   (theory 1/4 = {0.25:.4f})")
print(
    f"(SE of the three lines above: ± {sqrt(pa*(1-pa)/N):.4f}, ± {sqrt(pb*(1-pb)/N):.4f}, ± {sqrt(pc*(1-pc)/N):.4f} — binomial sqrt(p(1-p)/N) at N = {N:,}; the largest |measured - theory| is {max(abs(pa-1/3)/sqrt(pa*(1-pa)/N), abs(pb-0.5)/sqrt(pb*(1-pb)/N), abs(pc-0.25)/sqrt(pc*(1-pc)/N)):.1f} SE)"
)


=== §1 Bertrand's chord: three 'uniform' measures, three answers (unit circle, side = sqrt(3)) ===
(a) two uniform endpoints        P(L > sqrt3) = 0.3324   (theory 1/3 = 0.3333)
(b) uniform distance to centre   P(L > sqrt3) = 0.4994   (theory 1/2 = 0.5000)
(c) uniform midpoint in the disc P(L > sqrt3) = 0.2491   (theory 1/4 = 0.2500)
(SE of the three lines above: ± 0.0005, ± 0.0005, ± 0.0004 — binomial sqrt(p(1-p)/N) at N = 1,000,000; the largest |measured - theory| is 2.0 SE)


### §2 — Monty Hall as a conditioning statement

Staying wins $1/3$ of the time, switching $2/3$: conditioning on the host's door is a statement about which σ-algebra the host's
choice lives in, and the simulation returns the theorem's numbers.

In [3]:
sec(2, "Monty Hall as a conditioning statement")
N = 1_000_000
car = rng.integers(0, 3, N)
pick = rng.integers(0, 3, N)
ps = np.mean(car == pick)
print(f"stay wins   = {ps:.4f}   switch wins = {1-ps:.4f}   (theory 1/3, 2/3)")
print(
    f"(SE of the line above: ± {sqrt(ps*(1-ps)/N):.4f} for either frequency — binomial sqrt(p(1-p)/N) at N = {N:,})"
)


=== §2 Monty Hall as a conditioning statement ===


stay wins   = 0.3332   switch wins = 0.6668   (theory 1/3, 2/3)
(SE of the line above: ± 0.0005 for either frequency — binomial sqrt(p(1-p)/N) at N = 1,000,000)


### §6 — Borel–Kolmogorov: the same great circle, two conditional laws

A uniform point on the sphere conditioned on a great circle is uniform along it when the circle is reached as a latitude band,
and proportional to $\cos(\text{lat})$ when the same circle is reached as a longitude band. There is no "the" conditional law
given an event of probability zero — only a limit of events of positive probability, and the limit depends on the family.

In [4]:
sec(6, "Borel–Kolmogorov: conditioning a uniform point on the sphere on a great circle, two ways")
n = 4_000_000
v = rng.standard_normal((n, 3))
v /= np.linalg.norm(v, axis=1, keepdims=True)
lat = np.arcsin(v[:, 2])
lon = np.arctan2(v[:, 1], v[:, 0])
eps = 0.02
# (i) the great circle as the equator: band |lat| < eps -> distribution of longitude
sel = np.abs(lat) < eps
h, _ = np.histogram(lon[sel], bins=6, range=(-pi, pi))
print(f"(i) equator via latitude band:  longitude histogram (6 bins, normalised) = {h/h.sum()}")
# (ii) the same great circle as a meridian (rotate axes): band |lon| < eps -> distribution of latitude
sel = np.abs(lon) < eps
h, edges = np.histogram(lat[sel], bins=6, range=(-pi / 2, pi / 2))
centres = (edges[:-1] + edges[1:]) / 2
print(f"(ii) meridian via longitude band: latitude histogram (6 bins, normalised) = {h/h.sum()}")
print(
    f"     cos(latitude) at bin centres, normalised                             = {np.cos(centres)/np.cos(centres).sum()}"
)
print(
    "Same curve on the sphere, two conditional densities: uniform in (i), proportional to cos(lat) in (ii)."
)


=== §6 Borel–Kolmogorov: conditioning a uniform point on the sphere on a great circle, two ways ===


(i) equator via latitude band:  longitude histogram (6 bins, normalised) = [0.1659 0.1707 0.1645 0.1664 0.1645 0.168 ]
(ii) meridian via longitude band: latitude histogram (6 bins, normalised) = [0.0665 0.1833 0.2528 0.2468 0.1876 0.0631]
     cos(latitude) at bin centres, normalised                             = [0.067 0.183 0.25  0.25  0.183 0.067]
Same curve on the sphere, two conditional densities: uniform in (i), proportional to cos(lat) in (ii).


---
## Stop 2 — §2.2. Densities: absolute continuity and the Radon–Nikodym derivative

This stop quotes no section of `worked_numbers.py`; it is read, not run.

## Stop 3 — §2.3. Change of variables, and what a normalising flow computes

This stop quotes no section of `worked_numbers.py`; it is read, not run. (The Jacobian experiment lives in the companion's `A5`,
notebook 00.)

## Stop 4 — §2.4. Divergences as integrated density ratios, and the support problem

The text quotes `§20` twice (the table and the Pinsker check); both are lines of the one section below.

### §20 — Divergences under disjoint support

Two uniforms shifted by $\theta$: KL is infinite for every $\theta > 0$, JS saturates at $\ln 2$ once the supports stop overlapping, TV
reaches $1$, and only $W_1 = \theta$ keeps saying *how far*. The Pinsker check at the end relates the two families.

In [5]:
sec(20, "Divergences under disjoint support: U[0,1] vs U[theta, 1+theta]")
for th in (0.0, 0.01, 0.1, 0.5, 1.0, 2.0):
    ov = max(0.0, 1 - th)
    tv = 1 - ov
    js = log(2) if th >= 1 else None
    kl = 0.0 if th == 0 else float('inf')
    print(
        f"theta={th:<4}: KL = {kl},  TV = {tv:.2f},  W1 = {th:.2f}"
        + (f",  JS = ln2 = {js:.4f} (saturated)" if js else "")
    )
print(
    "Pinsker check on Bernoulli(0.5) vs Bernoulli(0.6): TV = 0.1, KL(nats) = %.5f, sqrt(KL/2) = %.4f >= TV"
    % (0.5 * log(0.5 / 0.6) + 0.5 * log(0.5 / 0.4), sqrt((0.5 * log(0.5 / 0.6) + 0.5 * log(0.5 / 0.4)) / 2))
)


=== §20 Divergences under disjoint support: U[0,1] vs U[theta, 1+theta] ===
theta=0.0 : KL = 0.0,  TV = 0.00,  W1 = 0.00
theta=0.01: KL = inf,  TV = 0.01,  W1 = 0.01
theta=0.1 : KL = inf,  TV = 0.10,  W1 = 0.10
theta=0.5 : KL = inf,  TV = 0.50,  W1 = 0.50
theta=1.0 : KL = inf,  TV = 1.00,  W1 = 1.00,  JS = ln2 = 0.6931 (saturated)
theta=2.0 : KL = inf,  TV = 1.00,  W1 = 2.00,  JS = ln2 = 0.6931 (saturated)
Pinsker check on Bernoulli(0.5) vs Bernoulli(0.6): TV = 0.1, KL(nats) = 0.02041, sqrt(KL/2) = 0.1010 >= TV


---
## Stop 5 — §3.3. The gradient of an expectation: two estimators, one theorem, and the loss where it fails

The text quotes `§33` (the moving-support case) first and `§18` (score function against pathwise on a fixed support) second.

### §33 — The score-function gradient when the support moves with the parameter

$X \sim U(0, \theta)$, $f(x) = x$, true derivative $1/2$: the naive score-function estimator gives $-0.5014 \pm 0.0006$ — the wrong
sign — because the Leibniz boundary term $f(\theta)\,p_\theta(\theta) = 1$ was dropped; adding it back gives $+0.4986$, and a
finite difference with common random numbers gives $+0.5014$.

In [6]:
sec(
    33,
    "Score-function gradient with a parameter-dependent support: X ~ U(0, theta), f(x) = x — the naive estimator gives -1/2, the boundary term 1 restores the true 1/2",
)
g33 = np.random.default_rng(2026_1016_33)
theta, n = 2.0, 200_000
U = g33.uniform(0, 1, n)
X = theta * U
print(f"true d/dtheta E[X] = d/dtheta (theta/2) = {0.5:.4f}")
naive = X * (-1 / theta)  # f(x) * d/dtheta log p_theta(x) = x * (-1/theta)
print(
    f"naive score-function estimate E[X * d/dtheta log p_theta(X)] = E[-X/theta] = {naive.mean():+.4f}   (SE {naive.std(ddof=1)/sqrt(n):.4f}; theory -1/2)"
)
boundary = theta * (1 / theta)  # f(theta) * p_theta(theta) * d(theta)/d(theta)
print(f"boundary term f(theta) p_theta(theta) = theta * (1/theta) = {boundary:.4f}")
print(f"corrected = naive + boundary = {naive.mean()+boundary:+.4f}   (theory +0.5000)")
h = 0.01
fd = (np.mean((theta + h) * U) - np.mean((theta - h) * U)) / (
    2 * h
)  # common random numbers: same U at theta +/- h
print(f"finite-difference check with common random numbers, theta ± {h}: {fd:+.4f}   (theory +0.5000)")


=== §33 Score-function gradient with a parameter-dependent support: X ~ U(0, theta), f(x) = x — the naive estimator gives -1/2, the boundary term 1 restores the true 1/2 ===
true d/dtheta E[X] = d/dtheta (theta/2) = 0.5000
naive score-function estimate E[X * d/dtheta log p_theta(X)] = E[-X/theta] = -0.5014   (SE 0.0006; theory -1/2)
boundary term f(theta) p_theta(theta) = theta * (1/theta) = 1.0000
corrected = naive + boundary = +0.4986   (theory +0.5000)
finite-difference check with common random numbers, theta ± 0.01: +0.5014   (theory +0.5000)


### §18 — Score function versus pathwise on a fixed support

$\partial_\mu\,\mathbb E[X^2]$ for $X \sim \mathcal N(\mu, 1)$, true value $2\mu = 3$: both estimators are unbiased, and the score-function
estimator's variance is $13$× the pathwise one's.

In [7]:
sec(
    18,
    "Gradient of an expectation: score-function vs pathwise estimators of d/dmu E[X^2], X ~ N(mu, 1), true value 2 mu",
)
mu, n = 1.5, 1000
reps = 2000
sf, pw = [], []
for _ in range(reps):
    eps = rng.standard_normal(n)
    x = mu + eps
    sf.append(np.mean(x**2 * (x - mu)))  # score function: f(x) * d/dmu log p(x)
    pw.append(np.mean(2 * x))  # pathwise: d/dmu (mu + eps)^2 = 2(mu+eps)
sf, pw = np.array(sf), np.array(pw)
print(
    f"true 2mu = {2*mu}; score-function: mean {sf.mean():.4f}, sd {sf.std():.4f};  pathwise: mean {pw.mean():.4f}, sd {pw.std():.4f};  variance ratio {sf.var()/pw.var():.1f}"
)
d18s, d18p = (sf - sf.mean()) ** 2, (
    pw - pw.mean()
) ** 2  # per-replication contributions to the two variances
R18 = sf.var() / pw.var()
cov18 = np.mean(
    (d18s - d18s.mean()) * (d18p - d18p.mean())
)  # the two estimators share each replication's draws, so their variance estimates covary
se_R18 = R18 * sqrt(
    (d18s.var() / sf.var() ** 2 + d18p.var() / pw.var() ** 2 - 2 * cov18 / (sf.var() * pw.var())) / reps
)  # joint delta method for a ratio of two correlated variance estimates
se_R18_naive = R18 * sqrt(
    (d18s.var() / sf.var() ** 2 + d18p.var() / pw.var() ** 2) / reps
)  # the same without the covariance term
print(
    f"(SE of the line above: the two means ± {sf.std()/sqrt(reps):.4f} and ± {pw.std()/sqrt(reps):.4f} (sd/sqrt({reps:,}));  the sds ± {se_sd(sf):.4f} and ± {se_sd(pw):.4f};  the variance ratio ± {se_R18:.1f} by the joint delta method — the two estimators use the same draws in each replication (correlation {np.corrcoef(sf, pw)[0, 1]:.2f}; of their squared deviations {np.corrcoef(d18s, d18p)[0, 1]:.2f}), so the ratio's bar carries their covariance; ignoring it would print ± {se_R18_naive:.1f})"
)


=== §18 Gradient of an expectation: score-function vs pathwise estimators of d/dmu E[X^2], X ~ N(mu, 1), true value 2 mu ===


true 2mu = 3.0; score-function: mean 3.0048, sd 0.2356;  pathwise: mean 3.0008, sd 0.0644;  variance ratio 13.4
(SE of the line above: the two means ± 0.0053 and ± 0.0014 (sd/sqrt(2,000));  the sds ± 0.0038 and ± 0.0010;  the variance ratio ± 0.4 by the joint delta method — the two estimators use the same draws in each replication (correlation 0.74; of their squared deviations 0.55), so the ratio's bar carries their covariance; ignoring it would print ± 0.6)


---
## Stop 6 — §4.2. The projection theorem, and the one result a practitioner uses every day

### §5 — Conditional expectation is an $L^2$ projection

The residual $Y - \mathbb E[Y \mid X]$ is orthogonal to every function of $X$ (four of them printed, each within noise of $0$), and the
mean-squared errors obey Pythagoras: the best linear predictor's MSE is the noise variance plus the squared distance from the
conditional mean to the linear fit.

In [8]:
sec(
    5, "Conditional expectation is an L2 projection: E[(Y - E[Y|X]) g(X)] = 0 for every g, and MSE decomposes"
)
n = 1_000_000
Xc = rng.standard_normal(n)
Y = Xc**2 + np.sin(3 * Xc) + 0.5 * rng.standard_normal(n)
m = Xc**2 + np.sin(3 * Xc)  # E[Y|X] exactly
resid = Y - m
for name, g in (("X", Xc), ("X^2", Xc**2), ("sin 3X", np.sin(3 * Xc)), ("1[X>0]", (Xc > 0).astype(float))):
    print(f"E[(Y - E[Y|X]) * {name:7s}] = {np.mean(resid*g):+.4f}")
f_lin = np.polyfit(Xc, Y, 1)
pred = np.polyval(f_lin, Xc)
print(f"MSE of best LINEAR predictor      = {np.mean((Y-pred)**2):.4f}")
print(f"MSE of E[Y|X]                     = {np.mean((Y-m)**2):.4f}   (= noise variance 0.25)")
print(
    f"E[(E[Y|X] - linear)^2]            = {np.mean((m-pred)**2):.4f}   -> sum {np.mean((Y-m)**2)+np.mean((m-pred)**2):.4f} (Pythagoras)"
)


=== §5 Conditional expectation is an L2 projection: E[(Y - E[Y|X]) g(X)] = 0 for every g, and MSE decomposes ===


E[(Y - E[Y|X]) * X      ] = -0.0002
E[(Y - E[Y|X]) * X^2    ] = +0.0003
E[(Y - E[Y|X]) * sin 3X ] = -0.0001
E[(Y - E[Y|X]) * 1[X>0] ] = -0.0004
MSE of best LINEAR predictor      = 2.7392
MSE of E[Y|X]                     = 0.2497   (= noise variance 0.25)


E[(E[Y|X] - linear)^2]            = 2.4885   -> sum 2.7382 (Pythagoras)


### §49 — Recalibration cannot buy sharpness

The Brier score splits into calibration plus refinement. A calibrated report $Q \in \{0.2, 0.8\}$ has refinement $0.16$; the constant $0.5$
is calibrated too, with refinement $0.25$ — coarsening kept calibration and lost sharpness. The canonical recalibration $c(q)$ of a
miscalibrated $q = 0.3 / 0.9$ removes its calibration term ($0.0100 \to 0$) and leaves its refinement exactly where it was; the simulated
lines repeat all of it on $N = 200{,}000$ draws, with standard errors.

In [9]:
sec(
    49,
    "Recalibration cannot buy sharpness: the calibration-refinement split of the Brier score for a calibrated two-level report, the constant 0.5, and the canonical recalibration of a miscalibrated report",
)
g49 = np.random.default_rng(2026_1016_49)
N49 = 200_000
print(
    "(a) exact, Q uniform on {0.2, 0.8}, Y ~ Bernoulli(Q): Brier = calibration E[(q - c(q))^2] + refinement E[c(q)(1 - c(q))]"
)
for name, lv, cv in (
    ("Q itself, calibrated       ", (0.2, 0.8), (0.2, 0.8)),
    ("constant 0.5, calibrated   ", (0.5, 0.5), (0.5, 0.5)),
    ("q = 0.3 / 0.9, miscalibrated", (0.3, 0.9), (0.2, 0.8)),
    ("its recalibration c(q)     ", (0.2, 0.8), (0.2, 0.8)),
):
    cal = 0.5 * sum((l - c) ** 2 for l, c in zip(lv, cv))
    ref49 = 0.5 * sum(c * (1 - c) for c in cv)
    print(f"(a)   {name}: calibration {cal:.4f}   refinement {ref49:.4f}   Brier {cal + ref49:.4f}")
Q = np.where(g49.integers(0, 2, N49) == 0, 0.2, 0.8)
Y = (g49.uniform(0, 1, N49) < Q).astype(float)


def brier_split(q, y, cmap=None):
    """calibration and refinement terms with c estimated per level of q (or taken from cmap), the Brier score, and SEs"""
    levels = np.unique(q)
    cal = ref49 = 0.0
    cq = np.empty_like(q)
    floor = 0.0
    for l in levels:
        s = q == l
        c = y[s].mean() if cmap is None else cmap[l]
        cq[s] = c
        pi_l = s.mean()
        cal += pi_l * (l - c) ** 2
        ref49 += pi_l * c * (1 - c)
        floor += c * (1 - c) / len(y)
    b = (q - y) ** 2
    r = (cq - y) ** 2
    return cal, ref49, b.mean(), b.std(ddof=1) / sqrt(len(y)), r.std(ddof=1) / sqrt(len(y)), floor, cq


for l in (0.2, 0.8):
    s = Q == l
    c = Y[s].mean()
    print(
        f"(b) simulated N = {N49:,}: level Q = {l}: {s.sum():,} cases, empirical c = {c:.4f} (SE {sqrt(c*(1-c)/s.sum()):.4f})"
    )
for name, q in (("Q itself    ", Q), ("constant 0.5", np.full(N49, 0.5))):
    cal, ref49, b, seb, ser, floor, _ = brier_split(q, Y)
    print(
        f"(b)   {name}: calibration {cal:.2e} (sampling floor sum_l c_l(1 - c_l)/N = {floor:.1e})   refinement {ref49:.4f} (SE {ser:.4f})   Brier {b:.4f} (SE {seb:.4f})"
    )
qm = np.where(Q == 0.2, 0.3, 0.9)
cal, ref49, b, seb, ser, floor, cq = brier_split(qm, Y)
print(
    f"(c) miscalibrated q = 0.3 / 0.9 on the same N = {N49:,}: calibration {cal:.4f}   refinement {ref49:.4f} (SE {ser:.4f})   Brier {b:.4f} (SE {seb:.4f})   (exact 0.0100 / 0.1600 / 0.1700)"
)
cal2, ref2, b2, seb2, ser2, _, _ = brier_split(cq, Y)
print(
    f"(c) canonical recalibration c(q), c estimated in-sample: levels {np.unique(cq).round(4).tolist()};  calibration {cal2:.1e}   refinement {ref2:.4f} (equal to q's: {ref2 == ref49})   Brier {b2:.4f} (SE {seb2:.4f})"
)
h = N49 // 2
cfit = {l: Y[:h][qm[:h] == l].mean() for l in (0.3, 0.9)}  # fit c on the first half, evaluate on the second
qh, yh = qm[h:], Y[h:]
cal_q, ref_q, b_q, _, _, _, _ = brier_split(qh, yh)
cqh = np.where(qh == 0.3, cfit[0.3], cfit[0.9])
cal_c, ref_c, b_c, seb_c, _, _, _ = brier_split(cqh, yh)
floor_c = sum(c * (1 - c) for c in cfit.values()) * (
    1 / h + 1 / (N49 - h)
)  # sum_l pi_l c_l(1-c_l)(1/n_fit,l + 1/n_hold,l), pi_l = 1/2, n_l ~ half of each half
print(
    f"(c) held out (c fitted on the first {h:,}, scored on the other {N49 - h:,}): c = {{0.3: {cfit[0.3]:.4f}, 0.9: {cfit[0.9]:.4f}}};  q: calibration {cal_q:.4f}, refinement {ref_q:.4f}, Brier {b_q:.4f};  c(q): calibration {cal_c:.1e} (sampling-noise expectation sum_l c_l(1 - c_l)(1/h + 1/(N - h)) = {floor_c:.1e}), refinement {ref_c:.4f}, Brier {b_c:.4f} (SE {seb_c:.4f})"
)
print(
    f"(c) recalibration moved the calibration term {cal_q:.4f} -> {cal_c:.1e} and left the refinement at {ref_q:.4f} = {ref_c:.4f};  coarsening Q to the constant 0.5 instead keeps calibration and raises refinement 0.16 -> 0.25"
)


=== §49 Recalibration cannot buy sharpness: the calibration-refinement split of the Brier score for a calibrated two-level report, the constant 0.5, and the canonical recalibration of a miscalibrated report ===
(a) exact, Q uniform on {0.2, 0.8}, Y ~ Bernoulli(Q): Brier = calibration E[(q - c(q))^2] + refinement E[c(q)(1 - c(q))]
(a)   Q itself, calibrated       : calibration 0.0000   refinement 0.1600   Brier 0.1600
(a)   constant 0.5, calibrated   : calibration 0.0000   refinement 0.2500   Brier 0.2500
(a)   q = 0.3 / 0.9, miscalibrated: calibration 0.0100   refinement 0.1600   Brier 0.1700
(a)   its recalibration c(q)     : calibration 0.0000   refinement 0.1600   Brier 0.1600
(b) simulated N = 200,000: level Q = 0.2: 99,708 cases, empirical c = 0.2004 (SE 0.0013)
(b) simulated N = 200,000: level Q = 0.8: 100,292 cases, empirical c = 0.7998 (SE 0.0013)
(b)   Q itself    : calibration 8.20e-08 (sampling floor sum_l c_l(1 - c_l)/N = 1.6e-06)   refinement 0.1602 (SE 0.0005)   Brier 0.

---
## Stop 7 — §5.1. The definitions and the implication diagram

### §7 — Modes of convergence: the typewriter sequence and escaping mass

The typewriter sequence converges to $0$ in probability and in $L^1$ while every sample point is hit infinitely often
($\limsup X_n = 1$ almost surely); escaping mass $X_n = n\,\mathbf 1[U < 1/n]$ converges to $0$ in probability while its expectation
stays $1$.

In [10]:
sec(7, "Modes of convergence: the typewriter sequence and escaping mass")
# typewriter: indicator of [k/2^m, (k+1)/2^m) enumerated; X_n(u) -> 0 in probability and in L1, but limsup = 1 for every u.
u = rng.uniform(0, 1, 100_000)
hits = np.zeros_like(u)
idx = 0
pm = []
for m in range(0, 12):
    for k in range(2**m):
        Xn = ((u >= k / 2**m) & (u < (k + 1) / 2**m)).astype(float)
        pm.append((idx, Xn.mean()))
        hits += Xn
        idx += 1
print(
    f"typewriter: P(X_n = 1) at n = 1, 3, 7, 1023, 4095: {[round(float(pm[i][1]),4) for i in (0,2,6,1022,4094)]}  -> 0"
)
print(
    f"typewriter: fraction of sample points u hit at least once per dyadic level (12 levels): {np.mean(hits >= 12):.4f}  -> limsup X_n = 1 a.s."
)
print(
    f"(SE of the P(X_n = 1) line: each entry is a frequency over the {len(u):,} points u, ± sqrt(p(1-p)/{len(u):,}) = "
    + ", ".join(f"{sqrt(pm[i][1]*(1-pm[i][1])/len(u)):.4f}" for i in (0, 2, 6, 1022, 4094))
    + "; the 1.0000 is exact — every u is hit at every level)"
)
# escaping mass: X_n = n * 1[U < 1/n]
esc7 = []
for nn in (10, 100, 1000, 10000):
    Xn = nn * (u < 1 / nn)
    print(
        f"escaping mass n={nn:>5}: P(X_n != 0) = {np.mean(Xn != 0):.4f}, E[X_n] = {Xn.mean():.3f}  (-> 0 in probability, E stays 1: no L1 convergence)"
    )
    esc7.append((nn, np.mean(Xn != 0)))
print(
    "(SE of the four lines above, from the same "
    + f"{len(u):,}"
    + " points: P(X_n != 0) ± "
    + ", ".join(f"{sqrt(q*(1-q)/len(u)):.4f}" for _, q in esc7)
    + ";  E[X_n] = n · P(X_n != 0) here, so ± "
    + ", ".join(f"{nn*sqrt(q*(1-q)/len(u)):.3f}" for nn, q in esc7)
    + " — the E[X_n] column is a mean of 100,000 terms of which only n^-1 of them are nonzero, which is why it wanders while its expectation is exactly 1)"
)


=== §7 Modes of convergence: the typewriter sequence and escaping mass ===


typewriter: P(X_n = 1) at n = 1, 3, 7, 1023, 4095: [1.0, 0.5014, 0.2523, 0.0021, 0.0006]  -> 0
typewriter: fraction of sample points u hit at least once per dyadic level (12 levels): 1.0000  -> limsup X_n = 1 a.s.
(SE of the P(X_n = 1) line: each entry is a frequency over the 100,000 points u, ± sqrt(p(1-p)/100,000) = 0.0000, 0.0016, 0.0014, 0.0001, 0.0001; the 1.0000 is exact — every u is hit at every level)
escaping mass n=   10: P(X_n != 0) = 0.0997, E[X_n] = 0.997  (-> 0 in probability, E stays 1: no L1 convergence)
escaping mass n=  100: P(X_n != 0) = 0.0104, E[X_n] = 1.039  (-> 0 in probability, E stays 1: no L1 convergence)
escaping mass n= 1000: P(X_n != 0) = 0.0011, E[X_n] = 1.110  (-> 0 in probability, E stays 1: no L1 convergence)
escaping mass n=10000: P(X_n != 0) = 0.0001, E[X_n] = 1.400  (-> 0 in probability, E stays 1: no L1 convergence)
(SE of the four lines above, from the same 100,000 points: P(X_n != 0) ± 0.0009, 0.0003, 0.0001, 0.0000;  E[X_n] = n · P(X_n != 0) here

---
## Stop 8 — §5.4. The law of the iterated logarithm: the envelope a loss curve actually lives in

### §8 — SLLN versus LIL on two million fair coins

How far the running mean strays, measured against the envelope $\sqrt{2\log\log n/n}\,\sigma$ and against the CLT's $1/\sqrt n$
band; the ratio between the two is $\sqrt{2\log\log n}$, about $2.3$ at $n = 10^6$.

In [11]:
sec(8, "SLLN vs LIL: how far a running mean of fair coins strays, against sqrt(2 log log n / n) * sigma")
n = 2_000_000
flips = rng.integers(0, 2, n)
S = np.cumsum(flips - 0.5)
ns = np.arange(1, n + 1)
mask = ns >= 100
env = 0.5 * np.sqrt(2 * np.log(np.log(ns[mask])) / ns[mask])  # sigma = 1/2 for a fair coin
dev = np.abs(S[mask]) / ns[mask]
print(f"fraction of n in [100, 2e6] with |S_n/n| > LIL envelope: {np.mean(dev > env):.5f}")
print(f"fraction of n in [100, 2e6] with |S_n/n| > 0.5*envelope : {np.mean(dev > 0.5*env):.4f}")
print(
    f"fraction of n in [100, 2e6] with |S_n/n| > 1/sqrt(n)*0.5 (a CLT-width band): {np.mean(dev > 0.5/np.sqrt(ns[mask])):.4f}"
)
for nn in (100, 1_000, 10_000, 100_000, 1_000_000):
    print(
        f"n={nn:>9}: envelope = {0.5*sqrt(2*log(log(nn))/nn):.5f}   1/sqrt(n) CLT scale = {0.5/sqrt(nn):.5f}   ratio = {sqrt(2*log(log(nn))):.3f}"
    )


=== §8 SLLN vs LIL: how far a running mean of fair coins strays, against sqrt(2 log log n / n) * sigma ===
fraction of n in [100, 2e6] with |S_n/n| > LIL envelope: 0.00007
fraction of n in [100, 2e6] with |S_n/n| > 0.5*envelope : 0.2615


fraction of n in [100, 2e6] with |S_n/n| > 1/sqrt(n)*0.5 (a CLT-width band): 0.3947
n=      100: envelope = 0.08738   1/sqrt(n) CLT scale = 0.05000   ratio = 1.748
n=     1000: envelope = 0.03109   1/sqrt(n) CLT scale = 0.01581   ratio = 1.966
n=    10000: envelope = 0.01054   1/sqrt(n) CLT scale = 0.00500   ratio = 2.107
n=   100000: envelope = 0.00350   1/sqrt(n) CLT scale = 0.00158   ratio = 2.211
n=  1000000: envelope = 0.00115   1/sqrt(n) CLT scale = 0.00050   ratio = 2.292


---
## Stop 9 — §7.6. The eval-set error bar, as a table

The fast path's destination: `§15` (the table), `§35` (McNemar on the paired comparison) and `§42` (what selecting the best of $m$
models on the same set costs, and which bound on it is a guarantee).

### §15 — Eval-set error bars, and the paired comparison

Hoeffding's $n$ per half-width; the binomial sd at $n = 1000$, 80% accuracy ($0.0126$, a 95% half-width of $2.5$ points); and the
paired comparison through the disagreement rate: a 10% disagreement rate with a 55/45 split is a one-point gap at about $1$ sd.

In [12]:
sec(
    15,
    "Eval-set error bars: Hoeffding n for a half-width eps at confidence 1-delta, and a binomial check at n=1000, acc=0.80",
)
for eps in (0.05, 0.02, 0.01, 0.005):
    for delta in (0.05,):
        print(
            f"eps={eps:<6} delta={delta}: Hoeffding n >= ln(2/delta)/(2 eps^2) = {math.ceil(log(2/delta)/(2*eps**2)):>8,}"
        )
acc, n = 0.80, 1000
se = sqrt(acc * (1 - acc) / n)
print(
    f"acc=0.80, n=1000: binomial sd = {se:.4f}; 95% normal interval half-width = {1.96*se:.4f}; Hoeffding half-width sqrt(ln(2/0.05)/(2n)) = {sqrt(log(40)/(2*n)):.4f}"
)
# two models, same test set, paired: variance depends on disagreement rate
dis = 0.10
p_win = 0.55  # of the disagreements, model A wins 55%
diff_sd = sqrt(
    dis * (1 - (2 * p_win - 1) ** 2 * dis) / n
)  # approx sd of (accA - accB) via paired disagreements
print(
    f"paired comparison, n=1000, disagreement rate 10%, A wins 55% of them: gap = {dis*(2*p_win-1):.4f}, sd ~ {diff_sd:.4f}  -> {dis*(2*p_win-1)/diff_sd:.2f} sd"
)


=== §15 Eval-set error bars: Hoeffding n for a half-width eps at confidence 1-delta, and a binomial check at n=1000, acc=0.80 ===
eps=0.05   delta=0.05: Hoeffding n >= ln(2/delta)/(2 eps^2) =      738
eps=0.02   delta=0.05: Hoeffding n >= ln(2/delta)/(2 eps^2) =    4,612
eps=0.01   delta=0.05: Hoeffding n >= ln(2/delta)/(2 eps^2) =   18,445
eps=0.005  delta=0.05: Hoeffding n >= ln(2/delta)/(2 eps^2) =   73,778
acc=0.80, n=1000: binomial sd = 0.0126; 95% normal interval half-width = 0.0248; Hoeffding half-width sqrt(ln(2/0.05)/(2n)) = 0.0429
paired comparison, n=1000, disagreement rate 10%, A wins 55% of them: gap = 0.0100, sd ~ 0.0100  -> 1.00 sd


### §35 — McNemar: the same one-point gap, three verdicts

No draws; the exact two-sided binomial tail (`binom_two_sided` is defined in the cell). On $n = 1000$ paired examples a one-point gap
is $55$ wins of $100$ discordant pairs ($p = 0.368$) or $10$ of $10$ ($p = 0.00195$): the discordant count, not the gap, is the sample.

In [13]:
sec(
    35,
    "McNemar on a paired evaluation: the same one-point gap at n = 1,000 is noise with 100 discordant pairs and decisive with 10",
)
from math import comb


def binom_two_sided(m, k):
    """exact two-sided p = 2 * P(Bin(m, 1/2) >= k) for k >= m/2, capped at 1"""
    return min(1.0, 2 * sum(comb(m, j) for j in range(k, m + 1)) / 2**m)


for m, k, note, n_eval in (
    (100, 55, "(i)", 1000),
    (10, 10, "(ii)", 1000),
    (160, 95, "(iii) exercise 7", 2000),
):  # exercise 7 scores a 2,000-example test set
    gap = (k - (m - k)) / n_eval * 100
    p = binom_two_sided(m, k)
    z = (k - m / 2) / sqrt(m / 4)
    print(
        f"{note:<17} discordant = {m:>3}, B wins {k:>3}: gap = {gap:.1f} points of {n_eval};  exact two-sided p = {p:.6f}"
        + (
            f";  normal-approximation z = {z:.3f}"
            if note != "(ii)"
            else f"  (= 2 * (1/2)^10 = {2*0.5**10:.6f})"
        )
    )


=== §35 McNemar on a paired evaluation: the same one-point gap at n = 1,000 is noise with 100 discordant pairs and decisive with 10 ===
(i)               discordant = 100, B wins  55: gap = 1.0 points of 1000;  exact two-sided p = 0.368202;  normal-approximation z = 1.000
(ii)              discordant =  10, B wins  10: gap = 1.0 points of 1000;  exact two-sided p = 0.001953  (= 2 * (1/2)^10 = 0.001953)
(iii) exercise 7  discordant = 160, B wins  95: gap = 1.5 points of 2000;  exact two-sided p = 0.021574;  normal-approximation z = 2.372


### §42 — Selecting the best of $m = 100$ fixed models: which bound is a guarantee

$p(1-p)$ is not a valid sub-Gaussian variance proxy for a Bernoulli(0.8) score ($1.10009 > 1.08329$), so $\sigma\sqrt{2\ln m} = 0.0384$
is not a guarantee: the best of $100$ models on $1{,}000$ examples exceeds it in $8.35\%$ of replications. The Hoeffding-valid
$\sqrt{\ln m/(2n)} = 0.0480$ is exceeded in $0.45\%$.

In [14]:
sec(
    42,
    "Selecting the best of m = 100 fixed models on n = 1,000 held-out examples: the variance-proxy bound sigma sqrt(2 ln m) is not a Hoeffding guarantee",
)
g42 = np.random.default_rng(2026_1016_42)
p, n42, m42, R42 = 0.8, 1000, 100, 2000
mgf = p * exp(-(1 - p)) + (1 - p) * exp(
    p
)  # E[exp(-(X - p))], X ~ Bernoulli(p), lambda = -1 (the lower tail of the accuracy)
print(
    f"(a) X ~ Bernoulli({p}): E[e^(-(X-{p}))] = {p} e^-{1-p:.1f} + {1-p:.1f} e^{p} = {mgf:.5f}  >  e^(Var(X)/2) = e^{p*(1-p)/2:.2f} = {exp(p*(1-p)/2):.5f}  -> p(1-p) is NOT a valid sub-Gaussian variance proxy;  Hoeffding's proxy 1/4 holds: e^(1/8) = {exp(1/8):.5f} >= {mgf:.5f}"
)
sigma = sqrt(p * (1 - p) / n42)
b_var, b_hoeff = sigma * sqrt(2 * log(m42)), sqrt(log(m42) / (2 * n42))
print(
    f"(b) n={n42:,}, m={m42}, p={p}: sigma = sqrt(p(1-p)/n) = {sigma:.5f};  variance-proxy bound sigma sqrt(2 ln m) = {b_var:.4f} (not a guarantee for Bernoulli)   Hoeffding-valid sqrt(ln m / (2n)) = {b_hoeff:.4f}"
)
acc = (
    g42.binomial(n42, p, (R42, m42)) / n42
)  # m independent models, each scored on its own n Bernoulli(p) correctness indicators
mx = (acc - p).max(1)
print(
    f"(c) {R42:,} replications: mean of max_j (accuracy_j - {p}) = {mx.mean():.4f} (MC SE {mx.std(ddof=1)/sqrt(R42):.4f};  median {np.median(mx):.4f}, 99th pct {np.percentile(mx, 99):.4f})   — under both {b_var:.4f} and {b_hoeff:.4f}; only the second is a proven bound on E[max]"
)
print(
    f"(c) fraction of replications whose max exceeds the variance-proxy bound {b_var:.4f}: {np.mean(mx > b_var):.4f};  exceeds the Hoeffding bound {b_hoeff:.4f}: {np.mean(mx > b_hoeff):.4f}"
)


=== §42 Selecting the best of m = 100 fixed models on n = 1,000 held-out examples: the variance-proxy bound sigma sqrt(2 ln m) is not a Hoeffding guarantee ===
(a) X ~ Bernoulli(0.8): E[e^(-(X-0.8))] = 0.8 e^-0.2 + 0.2 e^0.8 = 1.10009  >  e^(Var(X)/2) = e^0.08 = 1.08329  -> p(1-p) is NOT a valid sub-Gaussian variance proxy;  Hoeffding's proxy 1/4 holds: e^(1/8) = 1.13315 >= 1.10009
(b) n=1,000, m=100, p=0.8: sigma = sqrt(p(1-p)/n) = 0.01265;  variance-proxy bound sigma sqrt(2 ln m) = 0.0384 (not a guarantee for Bernoulli)   Hoeffding-valid sqrt(ln m / (2n)) = 0.0480
(c) 2,000 replications: mean of max_j (accuracy_j - 0.8) = 0.0312 (MC SE 0.0001;  median 0.0310, 99th pct 0.0440)   — under both 0.0384 and 0.0480; only the second is a proven bound on E[max]
(c) fraction of replications whose max exceeds the variance-proxy bound 0.0384: 0.0835;  exceeds the Hoeffding bound 0.0480: 0.0045


---
## Stop 10 — §8.2. Optional stopping, Ville's inequality, and the peeking analyst

### §22 — Peeking, and the test martingale that survives it

A $z$-test at fixed $n = 10{,}000$ rejects a fair coin 5% of the time; checked every 100 samples it rejects 38% of the time. The test
martingale stopped at $E_n \ge 20$ rejects under 5% of the time at every stopping rule (Ville).

In [15]:
sec(
    22,
    "Optional stopping: peeking at a z-test every 100 samples up to 10,000 vs a test martingale with Ville's inequality (alpha = 0.05)",
)
reps, N_, look = 4000, 10_000, 100
fp_peek = 0
fp_ville = 0
fp_fixed = 0
lam = 0.5
for _ in range(reps):
    x = rng.integers(0, 2, N_).astype(float)  # null: fair coin, mean 0.5
    S = np.cumsum(x - 0.5)
    ns = np.arange(1, N_ + 1)
    z = S[look - 1 :: look] / (0.5 * np.sqrt(ns[look - 1 :: look]))
    if np.any(np.abs(z) > 1.96):
        fp_peek += 1
    if abs(z[-1]) > 1.96:
        fp_fixed += 1
    # test martingale: product of (1 + lam*(2x-1)) with lam in (-1,1) -> nonneg martingale with E=1 under null; two-sided via mixture of +lam/-lam
    E_plus = np.cumprod(1 + lam * (2 * x - 1))
    E_minus = np.cumprod(1 - lam * (2 * x - 1))
    E = 0.5 * (E_plus + E_minus)
    if np.any(E >= 1 / 0.05):
        fp_ville += 1
print(f"fixed-n z-test at n=10,000:            false positive rate {fp_fixed/reps:.4f}")
print(f"z-test peeked every 100 (100 looks):   false positive rate {fp_peek/reps:.4f}")
print(
    f"test martingale, stop when E_n >= 20:  false positive rate {fp_ville/reps:.4f}   (Ville: <= 0.05 at EVERY stopping rule)"
)


=== §22 Optional stopping: peeking at a z-test every 100 samples up to 10,000 vs a test martingale with Ville's inequality (alpha = 0.05) ===


fixed-n z-test at n=10,000:            false positive rate 0.0460
z-test peeked every 100 (100 looks):   false positive rate 0.3835
test martingale, stop when E_n >= 20:  false positive rate 0.0437   (Ville: <= 0.05 at EVERY stopping rule)


### §37 — Error bars on §22, and a martingale that bets on several $\lambda$ at once

The `§22` rates carry Monte Carlo standard errors ($0.0437 \pm 0.0032$ for the martingale); a mixture over eight branches crosses $20$
in $100\%$ of runs at $p = 0.6$ against $41\%$ for the fixed $\lambda = 0.5$, with a null false-positive rate of $0.0465$.

In [16]:
sec(
    37,
    "Monte Carlo standard errors for the §22 false-positive rates, and a mixture test martingale that bets on several lambdas at once",
)
g37 = np.random.default_rng(2026_1016_37)
R22 = 4000
for name, p in (
    ("fixed-n z-test", 0.0460),
    ("z-test peeked every 100", 0.3835),
    ("test martingale E_n >= 20", 0.0437),
):
    print(
        f"(a) §22 {name:<27} rate {p:.4f} at {R22:,} replications: SE = sqrt(p(1-p)/{R22}) = {sqrt(p*(1-p)/R22):.4f}"
    )
lams = (0.1, 0.2, 0.3, 0.5)
for p1 in (0.6, 0.55):
    print(
        f"(b) growth rate per step at P(1)={p1}: "
        + ", ".join(f"lambda={l}: {p1*log(1+l)+(1-p1)*log(1-l):+.4f}" for l in lams)
        + "   (p log(1+lambda) + (1-p) log(1-lambda); the optimum is lambda = 2p-1)"
    )
reps, N_, thresh = 2000, 10_000, 20.0


def run_martingales(p1):
    """returns (crossed, cross_time) for the 8-branch mixture and for the fixed lambda=0.5 two-sided (§22) martingale"""
    Emix = np.ones((8, reps))
    Efix = np.ones((2, reps))
    t_mix = np.full(reps, -1)
    t_fix = np.full(reps, -1)
    for t in range(N_):
        s = 2.0 * (g37.uniform(0, 1, reps) < p1) - 1.0  # ±1 with P(+1) = p1
        for j, l in enumerate(lams):
            Emix[2 * j] *= 1 + l * s
            Emix[2 * j + 1] *= 1 - l * s
        Efix[0] *= 1 + 0.5 * s
        Efix[1] *= 1 - 0.5 * s
        new = (t_mix < 0) & (Emix.mean(0) >= thresh)
        t_mix[new] = t + 1
        new = (t_fix < 0) & (Efix.mean(0) >= thresh)
        t_fix[new] = t + 1
    return t_mix, t_fix


t_mix, t_fix = run_martingales(0.5)
fp = np.mean(t_mix >= 0)
fpf = np.mean(t_fix >= 0)
print(
    f"(b) null P(1)=0.5, {reps:,} replications, n={N_:,}: mixture (8 branches) false-positive rate of 'stop when E_n >= 20' = {fp:.4f} (SE {sqrt(fp*(1-fp)/reps):.4f});  fixed lambda=0.5 (§22) = {fpf:.4f} (SE {sqrt(fpf*(1-fpf)/reps):.4f});  Ville bound 0.0500"
)
for p1 in (0.6, 0.55):
    t_mix, t_fix = run_martingales(p1)
    cm, cf = np.mean(t_mix >= 0), np.mean(t_fix >= 0)
    med_m = np.median(t_mix[t_mix >= 0]) if cm > 0 else float('nan')
    med_f = np.median(t_fix[t_fix >= 0]) if cf > 0 else float('nan')
    print(
        f"(b) alternative P(1)={p1}: mixture crosses 20 by n={N_:,} in {cm:.4f} of runs, median crossing time {med_m:.0f};  fixed lambda=0.5 crosses in {cf:.4f} of runs, median crossing time {med_f:.0f}"
    )
    print(
        f"    (SE of the line above: mixture fraction ± {sqrt(cm*(1-cm)/reps):.4f}, median crossing time ± {se_median(t_mix[t_mix >= 0]):.0f};  fixed lambda=0.5 fraction ± {sqrt(cf*(1-cf)/reps):.4f}, median ± {se_median(t_fix[t_fix >= 0]):.0f}   — binomial SEs from the same {reps:,} runs; median SEs are half the order-statistic 68% interval of the crossing times)"
    )


=== §37 Monte Carlo standard errors for the §22 false-positive rates, and a mixture test martingale that bets on several lambdas at once ===
(a) §22 fixed-n z-test              rate 0.0460 at 4,000 replications: SE = sqrt(p(1-p)/4000) = 0.0033
(a) §22 z-test peeked every 100     rate 0.3835 at 4,000 replications: SE = sqrt(p(1-p)/4000) = 0.0077
(a) §22 test martingale E_n >= 20   rate 0.0437 at 4,000 replications: SE = sqrt(p(1-p)/4000) = 0.0032
(b) growth rate per step at P(1)=0.6: lambda=0.1: +0.0150, lambda=0.2: +0.0201, lambda=0.3: +0.0147, lambda=0.5: -0.0340   (p log(1+lambda) + (1-p) log(1-lambda); the optimum is lambda = 2p-1)
(b) growth rate per step at P(1)=0.55: lambda=0.1: +0.0050, lambda=0.2: -0.0001, lambda=0.3: -0.0162, lambda=0.5: -0.0889   (p log(1+lambda) + (1-p) log(1-lambda); the optimum is lambda = 2p-1)


(b) null P(1)=0.5, 2,000 replications, n=10,000: mixture (8 branches) false-positive rate of 'stop when E_n >= 20' = 0.0465 (SE 0.0047);  fixed lambda=0.5 (§22) = 0.0485 (SE 0.0048);  Ville bound 0.0500


(b) alternative P(1)=0.6: mixture crosses 20 by n=10,000 in 1.0000 of runs, median crossing time 169;  fixed lambda=0.5 crosses in 0.4065 of runs, median crossing time 61
    (SE of the line above: mixture fraction ± 0.0000, median crossing time ± 6;  fixed lambda=0.5 fraction ± 0.0110, median ± 4   — binomial SEs from the same 2,000 runs; median SEs are half the order-statistic 68% interval of the crossing times)


(b) alternative P(1)=0.55: mixture crosses 20 by n=10,000 in 1.0000 of runs, median crossing time 771;  fixed lambda=0.5 crosses in 0.0930 of runs, median crossing time 34
    (SE of the line above: mixture fraction ± 0.0000, median crossing time ± 24;  fixed lambda=0.5 fraction ± 0.0065, median ± 3   — binomial SEs from the same 2,000 runs; median SEs are half the order-statistic 68% interval of the crossing times)


---
## Stop 11 — §8.3. Maximal inequality, Azuma–Hoeffding, and where McDiarmid comes from

This stop quotes no section of `worked_numbers.py`; it is read, not run.

## Stop 12 — §8.4. Convergence, Robbins–Siegmund, and what "SGD converges" means

`§21` (four step schedules on the E2 mixture), `§34` (what a constant step does and does not do) and `§40` (multiplicative noise:
almost-sure convergence while the second moment diverges).

### §21 — SGD on E2 is Robbins–Monro

Logistic regression on the `§4` mixture by streaming SGD under four step schedules, test error at $t = 10^2 \dots 2\times10^4$ against
the Bayes risk $\Phi(-1) = 0.1587$: $1/t$ satisfies Robbins–Monro and is the slowest; $1/\sqrt t$ violates it and is the best at every
horizon shown; the constant steps do not settle.

In [17]:
sec(
    21,
    "Robbins–Monro SGD on E2 (logistic loss, d=20, mu=1): step 1/t vs constant 0.1, excess risk over Bayes",
)
d, mu = 20, 1.0


def stream(n):
    yy = rng.integers(0, 2, n) * 2 - 1
    XX = rng.standard_normal((n, d))
    XX[:, 0] += mu * yy
    return XX, yy


Xte, yte = stream(50_000)


def run(step_fn, T=20_000):
    w = np.zeros(d)
    out = []
    XX, yy = stream(T)
    for t in range(T):
        x, yv = XX[t], yy[t]
        g = -yv * x / (1 + exp(yv * (x @ w)))
        w -= step_fn(t + 1) * g
        if (t + 1) in (100, 1000, 10000, 20000):
            out.append(((t + 1), np.mean(np.sign(Xte @ w) != yte)))
    return out


traj21 = (
    {}
)  # the four trajectories are kept so that the notebook's figure plots THESE runs, not a fresh replicate
for name, fn in (
    ("1/t", lambda t: 1.0 / t),
    ("1/sqrt t", lambda t: 1.0 / sqrt(t)),
    ("const 0.1", lambda t: 0.1),
    ("const 1.0", lambda t: 1.0),
):
    res = run(fn)
    traj21[name] = res
    print(
        f"step {name:>9}: "
        + "  ".join(f"t={t:>5} err={e:.4f}" for t, e in res)
        + f"   (Bayes {Phi(-mu):.4f})"
    )


=== §21 Robbins–Monro SGD on E2 (logistic loss, d=20, mu=1): step 1/t vs constant 0.1, excess risk over Bayes ===


step       1/t: t=  100 err=0.2457  t= 1000 err=0.2110  t=10000 err=0.1931  t=20000 err=0.1884   (Bayes 0.1587)


step  1/sqrt t: t=  100 err=0.1918  t= 1000 err=0.1652  t=10000 err=0.1614  t=20000 err=0.1608   (Bayes 0.1587)


step const 0.1: t=  100 err=0.1922  t= 1000 err=0.1864  t=10000 err=0.1749  t=20000 err=0.1830   (Bayes 0.1587)


step const 1.0: t=  100 err=0.2418  t= 1000 err=0.2199  t=10000 err=0.1818  t=20000 err=0.2760   (Bayes 0.1587)


### §34 — What a constant step does, and does not do, on the noisy quadratic

Exact gradient: geometric convergence. Additive noise: a floor $\alpha/(2-\alpha)$ — $0.3333$, $0.0526$, $0.0050$ at
$\alpha = 0.5, 0.1, 0.01$. Multiplicative noise that vanishes at the optimum: $|x_t|$ contracts like $e^{-0.1068\,t}$.

In [18]:
sec(
    34,
    "Constant-step SGD on the noisy quadratic f(x) = x^2/2: exact gradient converges, additive noise leaves a floor alpha sigma^2/(2 - alpha), noise that vanishes at the optimum converges a.s. only under the contraction condition E log|1 - alpha(1+zeta)| < 0",
)
g34 = np.random.default_rng(2026_1016_34)
x0, T, chains = 5.0, 20_000, 200
# (a) exact gradient g_t = x_t: x_{t+1} = (1 - alpha) x_t
alpha = 0.1
print(
    f"(a) exact gradient, alpha={alpha}: "
    + "   ".join(
        f"|x_{t}| = {abs(x0*(1-alpha)**t):.4e} (theory (1-alpha)^t * 5 = {x0*(1-alpha)**t:.4e})"
        for t in (10, 100, 1000)
    )
)
# (b) additive noise g_t = x_t + xi_t, xi ~ N(0,1): x_{t+1} = (1-alpha) x_t - alpha xi_t -> stationary var alpha^2/(1-(1-alpha)^2) = alpha/(2-alpha)
for alpha in (0.5, 0.1, 0.01):
    x = np.full(chains, x0)
    acc2 = 0.0
    acc1 = 0.0
    acc1c = np.zeros(chains)
    acc2c = np.zeros(chains)  # per-chain sums for the SE line (same iterates, no new draw)
    xi = g34.standard_normal((T, chains))
    for t in range(T):
        x = x - alpha * (x + xi[t])
        if t >= T // 2:
            acc1 += x.sum()
            acc2 += (x * x).sum()
            acc1c += x
            acc2c += x * x
    m = acc1 / (chains * (T - T // 2))
    v = acc2 / (chains * (T - T // 2)) - m * m
    print(
        f"(b) additive noise, alpha={alpha:<5}: stationary var of x_t over the last 10,000 steps x 200 chains = {v:.4f}   (theory alpha/(2-alpha) = {alpha/(2-alpha):.4f}; mean x_t {m:+.4f})"
    )
    vc = acc2c / (T - T // 2) - (acc1c / (T - T // 2)) ** 2
    print(
        f"    (SE of the line above: stationary var ± {vc.std(ddof=1)/sqrt(chains):.4f}   — SD across the {chains} independent chains of each chain's own last-10,000-step variance, divided by sqrt({chains}))"
    )
# (c) multiplicative noise g_t = x_t (1 + zeta_t), zeta ~ N(0, 0.25) (variance 0.25, sd 0.5): x_{t+1} = x_t (1 - alpha (1 + zeta_t))
alpha, zsd = 0.1, 0.5
zeta = g34.standard_normal((10_000, chains)) * zsd
logabs = np.log(x0) + np.cumsum(
    np.log(np.abs(1 - alpha * (1 + zeta))), axis=0
)  # log|x_t| tracked exactly (no underflow)
for t in (100, 1000, 10000):
    la = logabs[t - 1]
    med, p90 = np.median(la), np.percentile(la, 90)
    print(
        f"(c) multiplicative noise, alpha={alpha}, zeta sd {zsd}: t={t:>5}: median |x_t| = {exp(med):.3e} (log10 {med/log(10):+.2f}), 90th pct |x_t| = {exp(p90):.3e} (log10 {p90/log(10):+.2f})"
    )
contr = np.log(np.abs(1 - alpha * (1 + zeta)))
print(
    f"(c) expected log-contraction E log|1 - alpha(1 + zeta)| = {contr.mean():+.4f} per step (MC over {contr.size:,} draws; log(1-alpha) = {log(1-alpha):+.4f} without noise)  -> |x_t| ~ 5 exp({contr.mean():+.4f} t): noise that vanishes at the optimum converges a.s. only because this contraction condition E log|1 - alpha(1+zeta)| < 0 holds here (§40: zeta = ±20 makes it positive and the iterates diverge a.s.)"
)
_gx34 = np.linspace(-12, 12, 2_000_001)
_gw34 = np.exp(-(_gx34**2) / 2) / sqrt(2 * pi) * (_gx34[1] - _gx34[0])
print(
    f"    (SE of the line above: E log|1 - alpha(1 + zeta)| ± {contr.std(ddof=1)/sqrt(contr.size):.4f}   — sd/sqrt(n) from the same {contr.size:,} draws; exact value by Gaussian quadrature {float(np.sum(np.log(np.abs(1 - alpha*(1 + zsd*_gx34)))*_gw34)):+.4f})"
)


=== §34 Constant-step SGD on the noisy quadratic f(x) = x^2/2: exact gradient converges, additive noise leaves a floor alpha sigma^2/(2 - alpha), noise that vanishes at the optimum converges a.s. only under the contraction condition E log|1 - alpha(1+zeta)| < 0 ===
(a) exact gradient, alpha=0.1: |x_10| = 1.7434e+00 (theory (1-alpha)^t * 5 = 1.7434e+00)   |x_100| = 1.3281e-04 (theory (1-alpha)^t * 5 = 1.3281e-04)   |x_1000| = 8.7394e-46 (theory (1-alpha)^t * 5 = 8.7394e-46)


(b) additive noise, alpha=0.5  : stationary var of x_t over the last 10,000 steps x 200 chains = 0.3333   (theory alpha/(2-alpha) = 0.3333; mean x_t +0.0002)


    (SE of the line above: stationary var ± 0.0004   — SD across the 200 independent chains of each chain's own last-10,000-step variance, divided by sqrt(200))


(b) additive noise, alpha=0.1  : stationary var of x_t over the last 10,000 steps x 200 chains = 0.0526   (theory alpha/(2-alpha) = 0.0526; mean x_t -0.0005)
    (SE of the line above: stationary var ± 0.0002   — SD across the 200 independent chains of each chain's own last-10,000-step variance, divided by sqrt(200))


(b) additive noise, alpha=0.01 : stationary var of x_t over the last 10,000 steps x 200 chains = 0.0050   (theory alpha/(2-alpha) = 0.0050; mean x_t +0.0011)
    (SE of the line above: stationary var ± 0.0001   — SD across the 200 independent chains of each chain's own last-10,000-step variance, divided by sqrt(200))
(c) multiplicative noise, alpha=0.1, zeta sd 0.5: t=  100: median |x_t| = 1.169e-04 (log10 -3.93), 90th pct |x_t| = 2.297e-04 (log10 -3.64)
(c) multiplicative noise, alpha=0.1, zeta sd 0.5: t= 1000: median |x_t| = 2.447e-46 (log10 -45.61), 90th pct |x_t| = 2.036e-45 (log10 -44.69)
(c) multiplicative noise, alpha=0.1, zeta sd 0.5: t=10000: median |x_t| = 0.000e+00 (log10 -463.35), 90th pct |x_t| = 0.000e+00 (log10 -460.14)
(c) expected log-contraction E log|1 - alpha(1 + zeta)| = -0.1068 per step (MC over 2,000,000 draws; log(1-alpha) = -0.1054 without noise)  -> |x_t| ~ 5 exp(-0.1068 t): noise that vanishes at the optimum converges a.s. only because this contraction condit

    (SE of the line above: E log|1 - alpha(1 + zeta)| ± 0.0000   — sd/sqrt(n) from the same 2,000,000 draws; exact value by Gaussian quadrature -0.1069)


### §40 — Multiplicative noise: almost-sure convergence while $\mathbb E[x_t^2] \to \infty$

At $\zeta = \pm 5$ the multipliers $0.4 / 1.4$ give $\mathbb E\log|m| = -0.2899 < 0$ and $\mathbb E[m^2] = 1.06 > 1$: every one of $2{,}000$
paths is below $10^{-6}$ by $t = 200$ while the exact second moment is $1.06^t$. At $\pm 20$ every path diverges. The two exact
conditions are printed for the Gaussian `§34`(c) case as well, and additive noise at $\alpha = 1$ shows the variance formula fixing
the variance but not the shape.

In [19]:
sec(
    40,
    "Multiplicative-noise SGD on f(x) = x^2/2: a.s. convergence with E[x_t^2] -> infinity, and the two exact contraction conditions",
)
g40 = np.random.default_rng(2026_1016_40)
alpha, x0, paths, T40 = 0.1, 1.0, 2000, 200
# x_{t+1} = x_t (1 - alpha (1 + zeta_t)); zeta = ±5 equiprobable -> multipliers 0.4 / 1.4
m_lo, m_hi = 1 - alpha * (1 + 5), 1 - alpha * (1 - 5)
Elog, Em2 = 0.5 * (log(abs(m_lo)) + log(abs(m_hi))), 0.5 * (m_lo**2 + m_hi**2)
print(
    f"(a) zeta = ±5, alpha={alpha}, x_0={x0}: multipliers {m_lo:.1f} / {m_hi:.1f};  E log|m| = (1/2) ln(0.56) = {Elog:+.4f} < 0 (a.s. contraction)   E[m^2] = (0.16 + 1.96)/2 = {Em2:.2f} > 1 (mean-square blow-up)   [exact]"
)
mult = np.where(g40.integers(0, 2, (T40, paths)) == 0, m_lo, m_hi)
logabs = log(x0) + np.cumsum(np.log(np.abs(mult)), axis=0)  # log|x_t| tracked exactly, as §34(c)
print(
    f"(a) {paths:,} paths: "
    + "   ".join(
        f"t={t}: median |x_t| = {exp(np.median(logabs[t-1])):.2e}, frac |x_t| < 1e-6 = {np.mean(logabs[t-1] < log(1e-6)):.4f}"
        for t in (50, 100, 200)
    )
    + f"   (theory median ~ exp({Elog:+.4f} t))"
)
print(
    f"    (SE of the line above: "
    + "   ".join(
        f"t={t}: median within a factor exp(±{se_median(logabs[t-1]):.3f}), frac ± {sqrt(np.mean(logabs[t-1] < log(1e-6))*(1 - np.mean(logabs[t-1] < log(1e-6)))/paths):.4f}"
        for t in (50, 100, 200)
    )
    + f"   — order-statistic SE of the median on the log scale and binomial SEs, from the same {paths:,} paths)"
)
Em4 = 0.5 * (m_lo**4 + m_hi**4)  # E[m^4], for the exact SD of x_t^2
for t in (10, 20, 40):
    x2 = np.exp(2 * logabs[t - 1])
    print(
        f"(a) t={t:>2}: MC mean of x_t^2 over {paths:,} paths = {x2.mean():.3e}   (exact E[x_t^2] = 1.06^{t} = {Em2**t:.3e};  max path x_t^2 = {x2.max():.3e}, median {np.median(x2):.3e} — the mean is carried by rare paths, so the MC mean undershoots the exact value once those paths are rarer than 1 in {paths:,})"
    )
    print(
        f"    (SE of the line above: MC mean of x_t^2 ± {x2.std(ddof=1)/sqrt(paths):.3e}   — sd/sqrt({paths:,}) from the same paths;  the exact SE, sqrt((E[m^4]^t - 1.06^(2t))/{paths:,}) with E[m^4] = {Em4:.4f}, is {sqrt((Em4**t - Em2**(2*t))/paths):.3e}: the empirical SE understates by the same rare-path mechanism)"
    )
# (b) zeta = ±20 -> multipliers -1.1 / 2.9
m_lo, m_hi = 1 - alpha * (1 + 20), 1 - alpha * (1 - 20)
Elog20 = 0.5 * (log(abs(m_lo)) + log(abs(m_hi)))
mult = np.where(g40.integers(0, 2, (100, paths)) == 0, m_lo, m_hi)
logabs20 = log(x0) + np.cumsum(np.log(np.abs(mult)), axis=0)
print(
    f"(b) zeta = ±20, alpha={alpha}: multipliers {m_lo:.1f} / {m_hi:.1f};  E log|m| = (1/2)(ln 1.1 + ln 2.9) = {Elog20:+.4f} > 0;  frac of {paths:,} paths with |x_100| > 1e6 = {np.mean(logabs20[99] > log(1e6)):.4f}  (median |x_100| = {exp(np.median(logabs20[99])):.2e})"
)
f20 = np.mean(logabs20[99] > log(1e6))
print(
    f"    (SE of the line above: frac ± {sqrt(f20*(1-f20)/paths):.4f} (binomial, from the same {paths:,} paths: every path exceeded 1e6);  median |x_100| within a factor exp(±{se_median(logabs20[99]):.3f}) (order-statistic SE on the log scale; exact median exp(100 E log|m|) = {exp(100*Elog20):.2e}))"
)
# (c) the two exact scalar conditions; Gaussian case by deterministic quadrature on the §32 grid (zeta = 0.5 g, g ~ N(0,1))
_gx = np.linspace(-12, 12, 2_000_001)
_gw = np.exp(-(_gx**2) / 2) / sqrt(2 * pi) * (_gx[1] - _gx[0])
mg = 1 - alpha * (1 + 0.5 * _gx)
rows = (
    ("N(0, 0.5^2) [§34(c)]", float(np.sum(np.log(np.abs(mg)) * _gw)), float(np.sum(mg**2 * _gw))),
    ("±5                 ", Elog, Em2),
    ("±20                ", Elog20, 0.5 * ((1 - alpha * 21) ** 2 + (1 - alpha * (-19)) ** 2)),
)
print(
    f"(c) alpha={alpha}: a.s. contraction iff E log|1 - alpha(1+zeta)| < 0;  mean-square contraction iff E[(1 - alpha(1+zeta))^2] < 1   (no noise: log(1-alpha) = {log(1-alpha):+.4f}, (1-alpha)^2 = {(1-alpha)**2:.4f})"
)
for name, el, em in rows:
    print(
        f"(c)   zeta ~ {name}:  E log|m| = {el:+.4f} ({'a.s. converges' if el < 0 else 'a.s. diverges '})   E[m^2] = {em:.4f} ({'L2 converges' if em < 1 else 'L2 diverges '})"
    )
# (d) additive noise, alpha = 1, xi = ±1: x_{t+1} = x_t - alpha (x_t + xi_t) = -xi_t
xi = np.where(g40.integers(0, 2, 10_000) == 0, -1.0, 1.0)
xd = np.empty(10_000)
x = x0
for t in range(10_000):
    x = x - 1.0 * (x + xi[t])
    xd[t] = x
vals, cnts = np.unique(xd, return_counts=True)
print(
    f"(d) additive noise, alpha=1, xi = ±1, 10,000 steps from x_0={x0}: x_{{t+1}} = -xi_t;  empirical support {{{', '.join(f'{v:+.0f}' for v in vals)}}} with frequencies {np.round(cnts/cnts.sum(), 4).tolist()}  — a two-point law, not Gaussian (excess kurtosis {np.mean(xd**4)/np.mean(xd**2)**2 - 3:+.1f} vs 0 for a Gaussian)"
)
p40 = np.mean(xd == 1)
print(
    f"    (SE of the line above: each frequency ± {sqrt(p40*(1-p40)/len(xd)):.4f} (binomial at p = 1/2, from the same 10,000 steps);  the excess kurtosis -2 is exact for any ±1 sample (x^4 = x^2 = 1), no SE)"
)
print(
    f"(d) stationary variance: empirical {xd.var():.4f}   (theory alpha sigma^2/(2 - alpha) = 1 * 1/(2 - 1) = {1.0*1.0/(2-1.0):.4f} still holds — the formula fixes the variance, not the shape)"
)
print(
    f"    (SE of the line above: empirical variance ± {se_var(xd):.4f}   — sqrt((m4 - s^4)/n) from the same 10,000 steps; for a ±1 sample s^2 = 1 - mean^2, so the SE is 2 |mean| sqrt(p(1-p)/n) up to O(1/n))"
)


=== §40 Multiplicative-noise SGD on f(x) = x^2/2: a.s. convergence with E[x_t^2] -> infinity, and the two exact contraction conditions ===
(a) zeta = ±5, alpha=0.1, x_0=1.0: multipliers 0.4 / 1.4;  E log|m| = (1/2) ln(0.56) = -0.2899 < 0 (a.s. contraction)   E[m^2] = (0.16 + 1.96)/2 = 1.06 > 1 (mean-square blow-up)   [exact]
(a) 2,000 paths: t=50: median |x_t| = 5.07e-07, frac |x_t| < 1e-6 = 0.5510   t=100: median |x_t| = 2.57e-13, frac |x_t| < 1e-6 = 0.9945   t=200: median |x_t| = 6.59e-26, frac |x_t| < 1e-6 = 1.0000   (theory median ~ exp(-0.2899 t))
    (SE of the line above: t=50: median within a factor exp(±0.000), frac ± 0.0111   t=100: median within a factor exp(±0.000), frac ± 0.0017   t=200: median within a factor exp(±0.000), frac ± 0.0000   — order-statistic SE of the median on the log scale and binomial SEs, from the same 2,000 paths)
(a) t=10: MC mean of x_t^2 over 2,000 paths = 1.210e+00   (exact E[x_t^2] = 1.06^10 = 1.791e+00;  max path x_t^2 = 8.367e+02, median 3.033e-

(d) additive noise, alpha=1, xi = ±1, 10,000 steps from x_0=1.0: x_{t+1} = -xi_t;  empirical support {-1, +1} with frequencies [0.4967, 0.5033]  — a two-point law, not Gaussian (excess kurtosis -2.0 vs 0 for a Gaussian)
    (SE of the line above: each frequency ± 0.0050 (binomial at p = 1/2, from the same 10,000 steps);  the excess kurtosis -2 is exact for any ±1 sample (x^4 = x^2 = 1), no SE)
(d) stationary variance: empirical 1.0000   (theory alpha sigma^2/(2 - alpha) = 1 * 1/(2 - 1) = 1.0000 still holds — the formula fixes the variance, not the shape)
    (SE of the line above: empirical variance ± 0.0001   — sqrt((m4 - s^4)/n) from the same 10,000 steps; for a ±1 sample s^2 = 1 - mean^2, so the SE is 2 |mean| sqrt(p(1-p)/n) up to O(1/n))


---
## Stop 13 — §12. Where intuition breaks, and what to do instead

This stop quotes no section of `worked_numbers.py`; it is a catalogue that points back to the sections above.

## Stop 14 — §13. What to actually take away

This stop quotes no section of `worked_numbers.py`; the numbers it relies on are the ones printed above.

---
*This notebook accompanies the Artifocial content series. Read the flagship at
[artifocial.com/blog/probability-domain-2026-oct-22](https://artifocial.com/blog/probability-domain-2026-oct-22), and see
what we build with these ideas at [artifocial.com/apps](https://artifocial.com/apps).*